In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'openset'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    df =pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_5": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 1096/1096 [00:00<00:00, 120392.77it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,metric,value
6488,0.1,0.25,20NG,ADB,0,ACC,53.8000
6489,0.1,0.25,20NG,ADB,0,F1,50.7301
6490,0.1,0.25,20NG,ADB,0,K-F1,48.3731
6491,0.1,0.25,20NG,ADB,0,N-F1,62.5150
6556,0.1,0.50,20NG,ADB,0,ACC,60.3900
...,...,...,...,...,...,...,...
12487,0.1,0.50,thucnews,clap,0,N-F1,0.0000
12512,0.1,0.75,thucnews,clap,0,ACC,50.3200
12513,0.1,0.75,thucnews,clap,0,F1,54.4698
12514,0.1,0.75,thucnews,clap,0,K-F1,0.0000


In [2]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                    0.1                                    \
known_cls_ratio                 0.25                              0.50   
metric                           ACC       F1     K-F1     N-F1    ACC   
dataset       method fold_idx                                            
20NG          ADB    0         53.80  50.7301  48.3731  62.5150  60.39   
                     1         40.82  40.1574  39.6174  42.8571  66.04   
                     2         22.19  25.5297  27.3626  16.3651  48.42   
                     3         51.45  56.4627  56.9910  53.8210  62.88   
                     4         58.17  49.3235  45.7212  67.3349  80.50   
...                              ...      ...      ...      ...    ...   
stackoverflow clap   2         33.90  25.1961   0.0000   0.0000  39.74   
                     3          9.85  13.1427   0.0000   0.0000  46.74   
                     4         25.20  23.7743   0.0000   0.0000  42.58   
thucnews      ab     0         32.70  26.7000  22.6000  43.1000  34.20   
              clap   0         22.94  32.5598   0.0000   0.0000  35.19   

labeled_ratio                                                             ...  \
known_cls_ratio                                            0.75           ...   
metric                              F1     K-F1     N-F1    ACC       F1  ...   
dataset       method fold_idx                                             ...   
20NG          ADB    0         69.6270  71.6784  49.1130  75.92  82.8313  ...   
                     1         67.4986  67.7088  65.3968  75.99  82.3294  ...   
                     2         57.5567  60.1148  31.9749  73.44  80.9509  ...   
                     3         69.5776  71.0790  54.5631  94.22  95.7202  ...   
                     4         81.5961  81.6761  80.7968  88.90  90.8816  ...   
...                                ...      ...      ...    ...      ...  ...   
stackoverflow clap   2         48.2263   0.0000   0.0000  67.45  70.7424  ...   
                     3         47.3287   0.0000   0.0000  68.05  64.6248  ...   
                     4         50.0485   0.0000   0.0000  69.97  69.2000  ...   
thucnews      ab     0         28.3000  26.2000  43.5000  27.10  24.8000  ...   
              clap   0         43.5344   0.0000   0.0000  50.32  54.4698  ...   

labeled_ratio                      1.0                                    \
known_cls_ratio                   0.25            0.50                     
metric                            K-F1     N-F1    ACC       F1     K-F1   
dataset       method fold_idx                                              
20NG          ADB    0         67.5778  69.8795  76.26  81.8210  82.8430   
                     1         74.2382  80.5954  86.62  89.4433  89.8371   
                     2         61.4170  56.7916  75.05  81.3614  82.7434   
                     3         67.2164  66.7866  85.88  88.5902  89.1119   
                     4         75.9690  85.0974  87.90  90.5715  91.0527   
...                                ...      ...    ...      ...      ...   
stackoverflow clap   2          0.0000   0.0000  59.22  56.7167   0.0000   
                     3          0.0000   0.0000  63.66  55.7336   0.0000   
                     4          0.0000   0.0000  64.91  61.3738   0.0000   
thucnews      ab     0             NaN      NaN    NaN      NaN      NaN   
              clap   0             NaN      NaN    NaN      NaN      NaN   

labeled_ratio                                                             
known_cls_ratio                          0.75                             
metric                            N-F1    ACC       F1     K-F1     N-F1  
dataset       method fold_idx                                             
20NG          ADB    0         71.6010  86.95  91.9988  93.4491  70.2454  
                     1         85.5051  80.50  87.5089  90.8249  37.7682  
                     2         67.5416  90.32  94.2404  95.3900  76.9968  
       

In [3]:
# 基于 df_melted = [labeled_ratio, known_cls_ratio, dataset, method, fold_idx, metric, value]

# 认为：只要某个 metric 有记录，就视为该 (dataset, method, labeled_ratio, known_cls_ratio, fold_idx) 已测试
cells = (df_melted[["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"]]
         .drop_duplicates()
         .sort_values(["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"])
         .reset_index(drop=True))

# 1) 每个 dataset × method 的维度覆盖与计数
progress_by_dm = (cells
    .groupby(["dataset","method"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),  # 完成的组合总数
    )
    .reset_index()
)

# 2) 每个 dataset × method × labeled_ratio × known_cls_ratio 下完成了多少个 fold
fold_coverage = (cells
    .groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset","method","labeled_ratio","known_cls_ratio"])
)

In [4]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio                      0.1                                    \
known_cls_ratio                   0.25             0.50             0.75   
dataset       method                                                       
20NG          ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              DeepUnk  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                ...              ...              ...   
stackoverflow PLM_OOD           [0, 3]           [0, 3]           [0, 3]   
              ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
thucnews      ab                   [0]              [0]              [0]   
              clap                 [0]              [0]              [0]   

labeled_ratio                      0.5                                    \
known_cls_ratio                   0.25             0.50             0.75   
dataset       method                                                       
20NG          ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              DeepUnk     [0, 1, 2, 3]        [0, 1, 2]              [0]   
              DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                ...              ...              ...   
stackoverflow PLM_OOD           [0, 3]           [0, 3]           [0, 3]   
              ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
              clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
thucnews      ab                   NaN              NaN              NaN   
              clap                 NaN              NaN              NaN   

labeled_ratio                      1.0                                    
known_cls_ratio                   0.25             0.50             0.75  
dataset       method                                                      
20NG          ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
              DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
              DeepUnk     [0, 1, 2, 3]        [0, 1, 2]              [0]  
              DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
              KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
...                                ...              ...              ...  
stackoverflow PLM_OOD           [0, 3]           [0, 3]              [3]  
              ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
              clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
thucnews      ab                   NaN              NaN              NaN  
              clap                 NaN              NaN              NaN  

[88 rows x 9 columns]

In [5]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.7854096520763187

In [6]:
all_num

4455

In [7]:
cur_num

3499.0